# Cloud Project: Production Data Engineering Platform


## Problem Statement

Design and implement a production-style data platform in the cloud.

System should:
- Ingest batch data from APIs
- Ingest streaming data via Kafka
- Store raw + processed data in S3 (data lake)
- Load structured data into PostgreSQL (RDS)
- Orchestrate workflows using Airflow
- Support analytics queries

Goal:
👉 Build a real-world data engineering system


## Architecture

```text
                ┌────────────┐
                │   API      │
                └────┬───────┘
                     │
                     ▼
                ┌────────────┐
                │   EC2      │
                │  (ETL)     │
                └────┬───────┘
                     │
        ┌────────────┼────────────┐
        ▼                         ▼
   ┌──────────┐             ┌────────────┐
   │    S3    │             │    RDS     │
   │ DataLake │             │ PostgreSQL │
   └──────────┘             └────────────┘

Streaming:
Producer → Kafka → Consumer → RDS

Orchestration:
Airflow → Runs ETL pipelines
```


## Batch Pipeline (Cloud ETL)

```python
import requests
import boto3
import psycopg2
import json
from datetime import datetime

s3 = boto3.client("s3")
BUCKET = "your-data-engineering-bucket"

def extract():
    return requests.get("https://jsonplaceholder.typicode.com/users").json()

def save_raw(data):
    filename = f"raw/users_{datetime.now().isoformat()}.json"
    
    with open("raw.json", "w") as f:
        json.dump(data, f)
    
    s3.upload_file("raw.json", BUCKET, filename)

def transform(data):
    return [
        {
            "id": d["id"],
            "name": d["name"].title(),
            "email": d["email"].lower()
        }
        for d in data
    ]

def save_processed(data):
    filename = f"processed/users_{datetime.now().isoformat()}.json"
    
    with open("processed.json", "w") as f:
        json.dump(data, f)
    
    s3.upload_file("processed.json", BUCKET, filename)

def load(data):
    conn = psycopg2.connect(
        host="your-rds-endpoint",
        database="de_course",
        user="admin",
        password="admin"
    )
    
    cur = conn.cursor()
    
    for d in data:
        cur.execute("""
        INSERT INTO users (id, name, email)
        VALUES (%s, %s, %s)
        ON CONFLICT (id) DO NOTHING
        """, (d["id"], d["name"], d["email"]))
    
    conn.commit()
    conn.close()
```


## Run Batch Pipeline

```python
def run_batch_pipeline():
    data = extract()
    save_raw(data)
    cleaned = transform(data)
    save_processed(cleaned)
    load(cleaned)
    
    print("Batch pipeline completed")
```


## Streaming Pipeline (Kafka -> RDS)

```python
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    "events-topic",
    bootstrap_servers="localhost:9092",
    value_deserializer=lambda x: json.loads(x.decode("utf-8"))
)

def streaming_pipeline():
    conn = psycopg2.connect(
        host="your-rds-endpoint",
        database="de_course",
        user="admin",
        password="admin"
    )
    cur = conn.cursor()

    for message in consumer:
        data = message.value
        
        cleaned = {
            "user_id": data["user_id"],
            "event": data["event"].upper(),
            "amount": int(data["amount"])
        }

        cur.execute("""
        INSERT INTO events (user_id, event, amount)
        VALUES (%s, %s, %s)
        """, (cleaned["user_id"], cleaned["event"], cleaned["amount"]))

        conn.commit()
```


## Airflow Integration (Concept)

```python
# DAG triggers batch pipeline daily

run_batch = PythonOperator(
    task_id="run_batch_pipeline",
    python_callable=run_batch_pipeline
)
```


## S3 Structure

s3://bucket/

raw/
processed/
analytics/

👉 Industry standard structure


## Analytics Queries

```python
conn = psycopg2.connect(
    host="your-rds-endpoint",
    database="de_course",
    user="admin",
    password="admin"
)

cur = conn.cursor()

cur.execute("""
SELECT COUNT(*) FROM users
""")

print(cur.fetchall())
```


## Enhancements

- Add logging system
- Add retry logic
- Add incremental loading
- Partition data in S3
- Use Parquet instead of JSON

Bonus:
- Add Spark processing
- Add dashboard layer


## Final Assignment

Upgrade system:

1. Add logging file
2. Add retry mechanism
3. Add incremental batch processing
4. Optimize storage (Parquet)

Bonus:
- Add Airflow DAG with multiple tasks


## How to Explain This Project

"I built a cloud-native data platform using AWS.

- Batch pipelines ingest API data into S3 and PostgreSQL
- Streaming pipelines process real-time events using Kafka
- Airflow orchestrates workflows
- Data is stored in a structured data lake
- System supports both batch and real-time analytics

This simulates a production-grade data engineering platform."


## Interview Questions

1. How do you design a cloud data platform?
2. Why use S3 + RDS?
3. How do you handle failures?
4. Batch vs streaming trade-offs?
5. How do you scale pipelines?


## Your tasks

- [ ] Build the full batch + streaming cloud pipeline skeleton.
- [ ] Ensure S3 raw/processed structure exists and is used.
- [ ] Connect to RDS and validate `users` + `events` analytics queries.
- [ ] Add at least one enhancement (logging/retry/incremental).
